# 选择 Chunk Size（第二节）

目标：在相同数据与评测问题集下，对比不同 `chunk_size`/`chunk_overlap` 的检索与回答质量，并得到一组可复用的默认参数。

## 0) 导入依赖并加载环境变量

In [1]:
from __future__ import annotations

import json
import random
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv("../.env")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


/tmp/ipykernel_2153307/1449842997.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 1) 读取文档

In [2]:
path = Path("../data/Understanding_Climate_Change.pdf")
assert path.exists(), f"找不到 PDF: {path.resolve()}"

docs = PyPDFLoader(str(path)).load()
len(docs)

33

## 2) 构造评测问题集

In [3]:
llm_q = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt_q = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Generate diverse evaluation questions based on the provided document excerpt. Return one question per line.",
        ),
        ("human", "Generate {n} questions.\n\nDocument excerpt:\n{excerpt}"),
    ]
)

# 对齐原版：用更大的文档范围生成题库，再随机抽取 k 个做评测
excerpt = "\n\n".join(d.page_content for d in docs[:20])

total_pool_questions = 50
num_eval_questions = 25

questions_text = (prompt_q | llm_q).invoke(
    {"n": total_pool_questions, "excerpt": excerpt}
).content

question_pool = [q.strip(" -\t") for q in questions_text.splitlines() if q.strip()]
eval_questions = random.sample(question_pool, k=min(num_eval_questions, len(question_pool)))

eval_questions[:5], len(eval_questions)

(['How do boreal forests contribute to carbon sequestration?',
  'How does methane contribute to the greenhouse effect compared to carbon dioxide?',
  'What relationship exists between warmer oceans and hurricanes?',
  'How do ice core samples help scientists understand climate history?',
  'What effects does ocean acidification have on coral reefs?'],
 25)

## 3) 定义评估函数（对每个 chunk size）

In [4]:
llm_answer = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_judge = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt_answer = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the question using only the provided context. If the answer is not in the context, say you don't know.",
        ),
        ("human", "Context:\n{context}\n\nQuestion: {question}"),
    ]
)

# 对齐原版：我们用 2 个指标来评估（faithfulness / relevancy），输出 JSON
prompt_judge = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an evaluator. Output JSON only with keys Faithfulness and Relevancy (0 or 1).\n"
            "- Faithfulness=1 if the answer is directly supported by the retrieved context, else 0.\n"
            "- Relevancy=1 if the retrieved context is relevant to the question, else 0.",
        ),
        (
            "human",
            "Question: {question}\n\nRetrieved Context:\n{context}\n\nAnswer:\n{answer}",
        ),
    ]
)


def _safe_parse_judge_json(text: str) -> dict:
    try:
        obj = json.loads(text)
        return {
            "Faithfulness": int(obj.get("Faithfulness", 0)),
            "Relevancy": int(obj.get("Relevancy", 0)),
        }
    except Exception:
        return {"Faithfulness": 0, "Relevancy": 0}


def evaluate_response_time_and_accuracy(
    chunk_size: int,
    eval_questions: list[str],
    *,
    k: int = 5,
    chunk_overlap: int | None = None,
):
    """返回 (avg_time_sec, avg_faithfulness, avg_relevancy, num_splits)。"""

    overlap = chunk_overlap if chunk_overlap is not None else chunk_size // 5

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        add_start_index=True,
    )
    splits = splitter.split_documents(docs)

    vectorstore = FAISS.from_documents(splits, OpenAIEmbeddings())
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})

    total_time = 0.0
    total_faithfulness = 0
    total_relevancy = 0

    for q in eval_questions:
        t0 = time.time()
        retrieved_docs = retriever.invoke(q)
        context = "\n\n".join(d.page_content for d in retrieved_docs)

        answer = (prompt_answer | llm_answer).invoke({"context": context, "question": q}).content
        judge_text = (prompt_judge | llm_judge).invoke(
            {"context": context, "question": q, "answer": answer}
        ).content

        total_time += time.time() - t0
        judge = _safe_parse_judge_json(judge_text)
        total_faithfulness += judge["Faithfulness"]
        total_relevancy += judge["Relevancy"]

    n = max(1, len(eval_questions))
    return (
        total_time / n,
        total_faithfulness / n,
        total_relevancy / n,
        len(splits),
        overlap,
    )

## 4) 测试不同 chunk size

In [5]:
chunk_sizes = [128, 256]

results = []
for chunk_size in chunk_sizes:
    avg_time, avg_faithfulness, avg_relevancy, num_splits, overlap = (
        evaluate_response_time_and_accuracy(chunk_size, eval_questions, k=5)
    )
    results.append(
        {
            "chunk_size": chunk_size,
            "chunk_overlap": overlap,
            "k": 5,
            "num_splits": num_splits,
            "avg_time_sec": avg_time,
            "avg_faithfulness": avg_faithfulness,
            "avg_relevancy": avg_relevancy,
        }
    )

    print(
        f"Chunk size {chunk_size} - Avg Response time: {avg_time:.2f}s, "
        f"Avg Faithfulness: {avg_faithfulness:.2f}, Avg Relevancy: {avg_relevancy:.2f} "
        f"(overlap={overlap}, splits={num_splits})"
    )

results

Chunk size 128 - Avg Response time: 17.77s, Avg Faithfulness: 0.84, Avg Relevancy: 1.00 (overlap=25, splits=795)
Chunk size 256 - Avg Response time: 19.05s, Avg Faithfulness: 0.92, Avg Relevancy: 1.00 (overlap=51, splits=364)


[{'chunk_size': 128,
  'chunk_overlap': 25,
  'k': 5,
  'num_splits': 795,
  'avg_time_sec': 17.768993072509765,
  'avg_faithfulness': 0.84,
  'avg_relevancy': 1.0},
 {'chunk_size': 256,
  'chunk_overlap': 51,
  'k': 5,
  'num_splits': 364,
  'avg_time_sec': 19.050749130249024,
  'avg_faithfulness': 0.92,
  'avg_relevancy': 1.0}]

In [6]:
# 汇总与排序（方便选择默认参数）
results_sorted = sorted(
    results,
    key=lambda r: (r["avg_relevancy"], r["avg_faithfulness"], -r["avg_time_sec"]),
    reverse=True,
)
results_sorted

[{'chunk_size': 256,
  'chunk_overlap': 51,
  'k': 5,
  'num_splits': 364,
  'avg_time_sec': 19.050749130249024,
  'avg_faithfulness': 0.92,
  'avg_relevancy': 1.0},
 {'chunk_size': 128,
  'chunk_overlap': 25,
  'k': 5,
  'num_splits': 795,
  'avg_time_sec': 17.768993072509765,
  'avg_faithfulness': 0.84,
  'avg_relevancy': 1.0}]